In [1]:
# ============================================================
# CELL 1 — Installs, imports, config, prompt
# ============================================================

!pip install openai openpyxl pandas numpy scipy -q

import json, time
import numpy as np
import pandas as pd
from scipy.stats import entropy as scipy_entropy
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('OPENAI_API'))
MODEL  = 'gpt-4o-mini'

# Same prompt used for justification generation
SYSTEM_PROMPT = (
    "You are a fact-checking analyst. "
    "Your role is to write a concise analytical note about a political claim — "
    "surfacing what it specifically says, what context is needed to understand it, "
    "and where it may be ambiguous — without judging whether it is true or false."
)

USER_TEMPLATE = """Write a short analytical note about the political claim below.

Guidelines:
- Engage directly with the specific content of the claim: the named person, the figure cited, the policy described, the comparison drawn. Make that content the subject of your note.
- If the claim attributes a statement or action to a named person, open by naming that person and what they said or did. Do not open with a generic description of the claim.
- Where a term or figure in the claim could be read in more than one way, say what those two readings are concretely — do not say that interpretation 'depends on' something without specifying what the two outcomes would be.
- You may use contrast ('but', 'however') to show two sides of the same element.
- Do not evaluate accuracy. Banned evaluative phrasings: 'correctly states', 'misleadingly claims', 'is accurate', 'is inaccurate', 'is true', 'is false'.
- Do not use procedural language: write the relevant considerations directly, do not describe what a fact-checker 'would need to' or 'should' verify.
- Maintain a neutral, analytical tone.

Strictly forbidden — never use any of the following words or phrases:
- "ambiguity", "ambiguous", "ambiguously"
- "key ambiguity", "central ambiguity", "core ambiguity"
- "hinges on", "turns on", "centers on"
- "The assertion", "The claim asserts", "The claim presents", "The claim suggests", "The claim states", "The claim implies"
- "The statement", "This statement"
- "Relevant context"
- "Additionally", "Furthermore", "Moreover"
- "Implicit assumption"
- "Understanding this claim requires", "Requires clarity on"
- "One must consider", "It is worth considering", "It is important to note"
- "depends on how", "depends on what", "depends on whether", "depends on the"
- "would need to be defined", "would need to be understood" "rather than"

Do not follow a fixed structure. Every note must differ from the others in how it is organised. Some notes open with the speaker's name; others with the specific figure or date. Some address a single pivot in two or three sentences; others trace a narrower point in one sentence. Never apply the same sentence-by-sentence template twice. Vary how each note is organised — in the opening, in the number of sentences, and in how you close. Do not always close by generalising about interpretation.

Length: Between 30 and 90 words. Match length strictly to complexity — a single-figure claim warrants 30–45 words; a claim with multiple interacting conditions warrants 70–90 words. Do not pad.

Output: A single paragraph. No bullet points, headers, or numbered lists.

Claim: {claim}

Analytical note:"""

print('✅ Cell 1 OK')

✅ Cell 1 OK


In [2]:
# ============================================================
# CELL 2 — Load dataset, extract stratified sample
# ============================================================

df = pd.read_excel('liar_plus_merged.xlsx')
df.columns = [c.strip() for c in df.columns]
print(f'Total rows: {len(df)}')

# Work on test set only
df_test = df[df['split'] == 'test'].copy()
print(f'Test set rows: {len(df_test)}')

# Binary label
false_labels = ['false', 'barely-true', 'pants-fire']
true_labels  = ['true', 'mostly-true', 'half-true']
df_test['binary_label'] = df_test['label'].apply(
    lambda x: 'false' if x in false_labels else 'true'
)
print('\nBinary label distribution in test set:')
print(df_test['binary_label'].value_counts())

# Stratified sample: 200 per class = 400 total
sample_size = 200
df_false = df_test[df_test['binary_label'] == 'false'].sample(n=sample_size, random_state=42)
df_true  = df_test[df_test['binary_label'] == 'true'].sample(n=sample_size, random_state=42)
df_sample = pd.concat([df_false, df_true]).reset_index(drop=True)

print(f'\nSample size: {len(df_sample)}')
print(df_sample['binary_label'].value_counts())

# Check paraphrase column is populated
missing_para = df_sample['gemini_paraphrase'].isna().sum()
print(f'\nMissing gemini_paraphrase in sample: {missing_para}')

print('\n✅ Cell 2 OK')

Total rows: 12791
Test set rows: 1267

Binary label distribution in test set:
binary_label
true     714
false    553
Name: count, dtype: int64

Sample size: 400
binary_label
false    200
true     200
Name: count, dtype: int64

Missing gemini_paraphrase in sample: 0

✅ Cell 2 OK


In [3]:
# ============================================================
# CELL 3 — Synchronous calls with logprobs on ORIGINAL claims
# ============================================================

def get_logprobs(claim_text):
    """Call GPT-4o-mini with logprobs=True and return token logprobs list."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': USER_TEMPLATE.format(claim=claim_text)},
            ],
            max_tokens=200,
            temperature=0.7,
            logprobs=True,
            top_logprobs=5,   # top 5 alternatives per token
        )
        token_logprobs = [
            t.logprob
            for t in response.choices[0].logprobs.content
        ]
        text = response.choices[0].message.content.strip()
        return token_logprobs, text
    except Exception as e:
        print(f'  ⚠️  Error: {e}')
        return None, None

# Run on original claims
results_original = []
print(f'Running {len(df_sample)} calls on ORIGINAL claims...')

for i, row in df_sample.iterrows():
    logprobs, text = get_logprobs(str(row['claim']))
    results_original.append({
        'idx':        i,
        'claim':      row['claim'],
        'label':      row['binary_label'],
        'logprobs':   logprobs,
        'text':       text,
    })
    if (len(results_original)) % 50 == 0:
        print(f'  {len(results_original)}/{len(df_sample)} done...')
    time.sleep(0.3)  # avoid rate limit

ok = sum(1 for r in results_original if r['logprobs'] is not None)
print(f'\n✅ Original calls completed: {ok}/{len(df_sample)}')

Running 400 calls on ORIGINAL claims...
  50/400 done...
  100/400 done...
  150/400 done...
  200/400 done...
  250/400 done...
  300/400 done...
  350/400 done...
  400/400 done...

✅ Original calls completed: 400/400


In [4]:
# ============================================================
# CELL 4 — Synchronous calls with logprobs on PARAPHRASED claims
# ============================================================

results_paraphrased = []
print(f'Running {len(df_sample)} calls on PARAPHRASED claims (Gemini paraphrase)...')

for i, row in df_sample.iterrows():
    para_claim = str(row['gemini_paraphrase'])
    logprobs, text = get_logprobs(para_claim)
    results_paraphrased.append({
        'idx':        i,
        'claim':      para_claim,
        'label':      row['binary_label'],
        'logprobs':   logprobs,
        'text':       text,
    })
    if (len(results_paraphrased)) % 50 == 0:
        print(f'  {len(results_paraphrased)}/{len(df_sample)} done...')
    time.sleep(0.3)

ok = sum(1 for r in results_paraphrased if r['logprobs'] is not None)
print(f'\n✅ Paraphrased calls completed: {ok}/{len(df_sample)}')

Running 400 calls on PARAPHRASED claims (Gemini paraphrase)...
  50/400 done...
  100/400 done...
  150/400 done...
  200/400 done...
  250/400 done...
  300/400 done...
  350/400 done...
  400/400 done...

✅ Paraphrased calls completed: 400/400


In [5]:
# ============================================================
# CELL 5 — Compute entropy & peakedness, compare, save
# ============================================================

import numpy as np
from scipy.stats import entropy as scipy_entropy

def compute_metrics(logprobs_list):
    """Given a list of token logprobs, compute mean entropy and peakedness."""
    if not logprobs_list:
        return None, None
    # Convert logprobs to probs
    probs = np.exp(logprobs_list)
    # Entropy: average per-token entropy (higher = more uncertain = less peaked)
    # Since we only have the top-1 logprob per token, we use -logprob as proxy
    mean_neg_logprob = -np.mean(logprobs_list)   # lower = more confident
    # Peakedness: average probability of chosen token (higher = more peaked)
    mean_prob        = np.mean(probs)             # higher = more confident
    return mean_neg_logprob, mean_prob

rows = []
for orig, para in zip(results_original, results_paraphrased):
    if orig['logprobs'] is None or para['logprobs'] is None:
        continue
    neg_lp_orig, prob_orig = compute_metrics(orig['logprobs'])
    neg_lp_para, prob_para = compute_metrics(para['logprobs'])

    rows.append({
        'label':              orig['label'],
        'neg_logprob_original':    neg_lp_orig,
        'neg_logprob_paraphrased': neg_lp_para,
        'delta_neg_logprob':       neg_lp_orig - neg_lp_para,
        'mean_prob_original':      prob_orig,
        'mean_prob_paraphrased':   prob_para,
        'delta_prob':              prob_orig - prob_para,
    })

df_ted = pd.DataFrame(rows)

print('=== TED RESULTS ===')
print(f'Valid pairs: {len(df_ted)}')
print()
print('--- Mean neg log-prob (lower = more confident/peaked) ---')
print(f'  Original:    {df_ted["neg_logprob_original"].mean():.4f}')
print(f'  Paraphrased: {df_ted["neg_logprob_paraphrased"].mean():.4f}')
print(f'  Delta:       {df_ted["delta_neg_logprob"].mean():.4f}')
print()
print('--- Mean token probability (higher = more peaked) ---')
print(f'  Original:    {df_ted["mean_prob_original"].mean():.4f}')
print(f'  Paraphrased: {df_ted["mean_prob_paraphrased"].mean():.4f}')
print(f'  Delta:       {df_ted["delta_prob"].mean():.4f}')
print()
print('--- By label ---')
print(df_ted.groupby('label')[['delta_neg_logprob','delta_prob']].mean().round(4))

# Save
df_ted.to_excel('TED_results.xlsx', index=False)
print('\n✅ Saved: TED_results.xlsx')

=== TED RESULTS ===
Valid pairs: 400

--- Mean neg log-prob (lower = more confident/peaked) ---
  Original:    14.8907
  Paraphrased: 14.0157
  Delta:       0.8750

--- Mean token probability (higher = more peaked) ---
  Original:    0.6541
  Paraphrased: 0.6612
  Delta:       -0.0071

--- By label ---
       delta_neg_logprob  delta_prob
label                               
false            -0.5512     -0.0046
true              2.3013     -0.0096

✅ Saved: TED_results.xlsx
